# Citi Bike acquisition and source audit

**Owner:** Tomislav Novosel. Implementation and initial explanations were prepared with AI assistance; see [the assistance record](../docs/AI_USE.md). Team review and subsequent contributions should be recorded as work continues.

**Task 7:** obtain and assemble the data with code. The working scope is the
official **2023 NYC archive**, covering a complete historical calendar year.
Separate `JC-*` Jersey City archives are outside this scope. This period is a
reproducible starting point for team review, not a claim that task 4 is agreed.

We download the annual ZIP and the following January 2024 ZIP, open any nested
monthly ZIPs and read **every** CSV. The adjacent archive captures December
starts that finished after New Year; extremely long rides ending after January
2024 remain a documented boundary limitation.
Reading only the first CSV would miss most trips in busy months. The raw archive
stays in ignored `data/raw/citibike/`; it is never included in Git.

Sources: [Citi Bike system data](https://citibikenyc.com/system-data) and
[the annual archive](https://s3.amazonaws.com/tripdata/2023-citibike-tripdata.zip) and
[the boundary archive](https://s3.amazonaws.com/tripdata/202401-citibike-tripdata.zip).


## Configuration and environment

Use the repository Python 3.12 environment. The source URL, SHA-256 fingerprint, working period and exploration cutoff live in one configuration file.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
ROOT = Path.cwd()
while not (ROOT / 'src/citibike.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open this notebook inside the repository folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from citibike import acquire_and_audit, sha256_file
config = json.loads((ROOT / 'citibike/source_config.json').read_text())
pd.set_option('display.max_colwidth', 110)

display(pd.Series(config, name='value').to_frame())

,value
year,2023
archives,"[{'url': 'https://s3.amazonaws.com/tripdata/2023-citibike-tripdata.zip', 'sha256': 'f5ac8d2b9556a4f1892556..."
scope,Published NYC archive; Jersey City JC-* archives are excluded.
period_reason,One complete historical calendar year with ride IDs and all seasons. Working analysis scope for team review.
boundary_policy,Include the following January archive to capture late-finished December starts. Aggregate by actual start ...
exploration_end,2023-09-30
candidate_holdout_start,2023-10-01


## Download, stream and reconcile

The helper downloads to a temporary `.part` file before accepting the ZIP.
Its SHA-256 must match the pinned source. Nested archives are handled in
temporary files; archive paths are not extracted into the repository.

CSV rows are processed in chunks of 200,000. All columns are read as strings
first to preserve identifiers and reveal schema differences. Timestamps are
parsed explicitly. Ride IDs are checked globally, including across different
chunks, CSVs and months; a duplicate is not silently counted twice.

The saved daily table contains **one row per printed NYC calendar date** and
one `trips` count. Gaps are reported instead of being filled with zero. The
sum of daily counts plus explicit out-of-year exclusions must equal the raw
record count. Archive labels do not guarantee every ride started in that year.
The cache is reused only
when the source fingerprint, audit version, year and output fingerprint match.
Set `reuse_audit=False` to re-read all source rows.


In [2]:
daily, audit = acquire_and_audit(reuse_audit=True)
controls = {
    'source_bytes': sum(item['bytes'] for item in audit['sources']),
    'raw_records': audit['rows'],
    'counted_rides': int(daily['trips'].sum()),
    'out_of_year_exclusions': audit['issues']['outside_selected_year'],
    'CSV_members': len(audit['csv_members']),
    'schema_variants': len(audit['schema_variants']),
    'expected_days': audit['expected_days'],
    'observed_days': len(daily),
    'missing_dates': len(audit['missing_dates']),
    'duplicate_extra_rows': audit['issues']['duplicate_ride_id_extra_rows'],
    'unparsed_start_times': audit['issues']['unparsed_start'],
}
display(pd.Series(controls, name='value').to_frame())
assert controls['raw_records'] == controls['counted_rides'] + controls['out_of_year_exclusions']
assert controls['observed_days'] == controls['expected_days']
assert controls['duplicate_extra_rows'] == 0
assert controls['unparsed_start_times'] == 0
for source in audit['sources']:
    print('Source:', source['url'], 'SHA-256:', source['sha256'])
print('Daily-table SHA-256:', audit['daily_sha256'])

Using verified cached archive: 2023-citibike-tripdata.zip


Using verified cached archive: 202401-citibike-tripdata.zip

Using matching audited table; reuse_audit=False repeats the full row audit.


,value
source_bytes,1971450658
raw_records,36995071
counted_rides,35107120
out_of_year_exclusions,1887951
CSV_members,42
schema_variants,1
expected_days,365
observed_days,365
missing_dates,0
duplicate_extra_rows,0


Source: https://s3.amazonaws.com/tripdata/2023-citibike-tripdata.zip SHA-256: f5ac8d2b9556a4f18925569681d60e960eae44ce7e2999c74726c8e1ff706c07
Source: https://s3.amazonaws.com/tripdata/202401-citibike-tripdata.zip SHA-256: 0a2e81eacd7bf3890712de8f2a1b56bda985c17d9e61b08acf5e7c7ec9f20eb0
Daily-table SHA-256: 5a129beca835f620c98c583d1367802aef57d3c6cf46e7385a55e2a553e6cd57


## File manifest and actual schemas

This manifest identifies every CSV read, its record count and date coverage. Schemas are compared by their actual column names and order; a changed schema needs an explicit mapping rather than guessing.

In [3]:
manifest = pd.DataFrame(audit['csv_members'])
manifest['file'] = manifest['member'].str.split('!/').str[-1]
display(manifest[['file', 'rows', 'first_started_at', 'last_started_at', 'distinct_start_dates']])
for number, schema in enumerate(audit['schema_variants'], start=1):
    print(f"Schema {number}: {len(schema['members'])} files")
    print(', '.join(schema['columns']))

,file,rows,first_started_at,last_started_at,distinct_start_dates
0,202301-citibike-tripdata_1.csv,1000000,2022-12-14 11:43:54.762000,2023-01-31 23:51:13.366000,31
1,202301-citibike-tripdata_2.csv,795405,2023-01-14 19:00:07.755000,2023-01-31 23:57:54.025000,18
2,202302-citibike-tripdata_1.csv,1000000,2023-01-02 17:04:53.302000,2023-02-28 23:55:22.868000,33
3,202302-citibike-tripdata_2.csv,696166,2023-02-14 19:00:02.455000,2023-02-28 23:56:43.541000,15
4,202303-citibike-tripdata_1.csv,1000000,2023-01-04 12:44:00.457000,2023-03-31 23:53:01.109000,37
5,202303-citibike-tripdata_2.csv,1000000,2023-03-14 20:00:02.076000,2023-03-31 23:58:12.686000,18
6,202303-citibike-tripdata_3.csv,118930,2023-03-14 20:00:08.585000,2023-03-31 23:56:49.332000,18
7,202304-citibike-tripdata_1.csv,1000000,2023-03-24 08:42:27.841000,2023-04-14 19:59:59.944000,22
8,202304-citibike-tripdata_2.csv,1000000,2023-03-31 10:17:16.459000,2023-04-30 23:57:00.875000,31
9,202304-citibike-tripdata_3.csv,749713,2023-04-14 20:00:01.762000,2023-04-30 23:56:36.146000,17


Schema 1: 42 files
ride_id, rideable_type, started_at, ended_at, start_station_name, start_station_id, end_station_name, end_station_id, start_lat, start_lng, end_lat, end_lng, member_casual


## Save compact evidence
The full raw ZIP and generated daily data remain ignored. A compact audit
manifest is versioned so a teammate can inspect the source and processing
results before downloading the archive. It is a recorded result; rebuilding
the notebook verifies it against the raw source again.


In [4]:
(ROOT / 'citibike/archive_manifest.json').write_text(
    json.dumps(audit, indent=2) + '\n', encoding='utf-8'
)
display(daily.head())
print('Daily table:', ROOT / 'data/processed/citibike_daily.csv')

,trips
date,
2023-01-01,50642
2023-01-02,58241
2023-01-03,51789
2023-01-04,74454
2023-01-05,71440


Daily table: C:\Users\tomis\OneDrive\THOMAS MORE APPLIED COMPUTER SCIENCE\YEAR 3\DL\CloudAI team repo\data\processed\citibike_daily.csv


## What this completes

Task 7 now has code-driven acquisition, a pinned source, an inventory of every
input CSV, verified coverage and a reproducible daily table. Run notebook 01
to explain the quality issues and daily-demand patterns (task 9).

No prediction model is trained here. Task 10 still needs a justified hypothesis
test; task 12 still needs the team's final evaluation dates and metrics.
